<a href="https://colab.research.google.com/github/gaatthishub/RP_Bot/blob/main/RP_Bot_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the required packages
!pip install -q accelerate
!pip install -q bitsandbytes>=0.43.0
!pip install -q transformers>=4.30.0
# Install necessary packages
!pip install praw
!pip install transformers
!pip install torch
!pip install sentencepiece
!pip install gradio

import re, time, json, random, sys, torch
from datetime import datetime
from typing import List, Optional

import praw
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from sentence_transformers import SentenceTransformer, util

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Connect to Google Drive for storage
from google.colab import drive
drive.mount('/content/drive')

# Create a project folder
!mkdir -p "/content/drive/MyDrive/rp_bot_project"

# Add the Google Drive path to Python's system path
sys.path.append('/content/drive/MyDrive/rp_bot_project')

# Import configuration
try:
  from config import *
  print("Loaded configuration from Google Drive")
except ImportError:
  # Fallback to empty defaults if config file doesn't exist
  print("No config.py found in Google Drive, using empty defaults")
  REDDIT_CLIENT_ID = ""
  REDDIT_CLIENT_SECRET = ""
  REDDIT_USERNAME = ""
  REDDIT_PASSWORD = ""
  HF_TOKEN = ""
  DEFAULT_SUBREDDIT = "RP_Bot_Testing"
  DEFAULT_AUDIENCE = "College students"
  DEFAULT_KEYWORDS = "AI, technology, education, regulation"
  DEFAULT_POSITION = "Random (default)"

Mounted at /content/drive
Loaded configuration from Google Drive


In [ ]:
# Login to Hugging Face
login(token=HF_TOKEN)

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_compute_dtype="bfloat16",
                         bnb_4bit_quant_type="nf4")

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Register <END> as an additional special token and make it the EOS
tokenizer.add_special_tokens({'eos_token': '<END>'})

model  = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",          # fits even on a T4
            quantization_config=bnb,
            low_cpu_mem_usage=True,     # streams weights
            trust_remote_code=True
         )
model.resize_token_embeddings(len(tokenizer))

emb_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")   # stays on CPU


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def get_voice_profile(audience, subreddit_name=None):
    """
    Select the appropriate voice profile based on audience and context.

    Parameters:
    - audience: Target audience (e.g., "College students", "Political conservatives")
    - subreddit_name: Optional name of subreddit for additional context

    Returns:
    - voice_instructions: Formatted guidelines for the language model
    """
    # Base voice profiles for different audiences
    nominal_wordcount = 200
    audience_voice_map = {
        "College students": {
            "style": "friendly",
            "length": nominal_wordcount,
            "tone": "casual and relatable",
            "examples": "campus life, courses, social events, study habits",
            "avoid": "complex regulatory scenarios, business jargon, technical policy details"
        },
        "Working professionals": {
            "style": "concise",
            "length": nominal_wordcount,
            "tone": "straightforward and practical",
            "examples": "workplace situations, career development, work-life balance",
            "avoid": "academic jargon, theoretical concepts without practical application"
        },
        "Political conservatives": {
            "style": "principled",
            "length": nominal_wordcount,
            "tone": "respectful and values-focused",
            "examples": "traditional values, individual responsibility, economic opportunities",
            "avoid": "condescending tone, overly academic language, assuming policy details"
        },
        "Political progressives": {
            "style": "community-focused",
            "length": nominal_wordcount,
            "tone": "inclusive and solutions-oriented",
            "examples": "community impact, equity considerations, systemic approaches",
            "avoid": "overly individualistic framing, dismissing structural factors"
        },
        "Environmental activists": {
            "style": "passionate",
            "length": nominal_wordcount,
            "tone": "urgent but hopeful",
            "examples": "tangible environmental impacts, community-level solutions",
            "avoid": "overly technical jargon, dismissing practical concerns"
        },
        "Tech enthusiasts": {
            "style": "informed",
            "length": nominal_wordcount,
            "tone": "knowledgeable but accessible",
            "examples": "current tech trends, hands-on applications, innovation",
            "avoid": "oversimplified analogies, outdated references"
        },
        "Parents": {
            "style": "thoughtful",
            "length": nominal_wordcount,
            "tone": "supportive and pragmatic",
            "examples": "family scenarios, educational choices, daily challenges",
            "avoid": "overly prescriptive advice, judgment of parenting styles"
        },
        "Retirees": {
            "style": "respectful",
            "length": nominal_wordcount,
            "tone": "clear and considerate",
            "examples": "life experience, community involvement, health and wellbeing",
            "avoid": "condescending 'explanations', assumptions about tech literacy"
        }
    }

    # Get the base profile for the audience
    profile = audience_voice_map.get(audience, audience_voice_map["College students"])

    # Adjust based on subreddit context if provided
    if subreddit_name:
        subreddit_lower = subreddit_name.lower()

        # Academic subreddits get slightly more formal tone
        if any(term in subreddit_lower for term in ["academic", "science", "research", "askhistorians"]):
            profile["style"] = "academic"
            profile["tone"] = "informed but accessible"
            profile["length"] += 30  # Allow slightly longer responses

        # Casual/meme subreddits get more conversational
        elif any(term in subreddit_lower for term in ["casual", "memes", "funny", "gaming"]):
            profile["style"] = "very casual"
            profile["tone"] = "lighthearted and conversational"
            profile["length"] -= 30  # Keep responses shorter

    # Format the voice instructions
    voice_instructions = f"""
    Use a {profile['tone']} tone.
    Keep your response under {profile['length']} words.
    Keep the MESSAGE section under {profile['length']} words (the UNSTATED_PREMISE line does **not** count toward this limit).
    Use examples related to {profile['examples']}.
    Avoid {profile['avoid']}.
    Structure your response in 1 paragraph at most. Use terminology common to this demographic.
    """
    print(f"The voice instructions are:{voice_instructions}")
    return voice_instructions, profile

In [ ]:
def generate_rhetorical_proxy(
    audience,
    topic,
    position,
    cultural_context,
    post_title="",
    post_content="",
    subreddit_name=None,
    max_new_tokens=256,
    temperature=0.7,
    extra_context="",
    style_tag="default"
):
    """
    Generate a persuasive message that uses enthymematic techniques.
    """

    if isinstance(temperature, str) and temperature == "auto":
        temperature = random.uniform(0.6, 0.9)
        top_p       = random.choice([0.9, 0.92, 0.95])
    else:
        top_p = 0.95

    # Get the voice profile based on audience and context
    voice_instructions, profile = get_voice_profile(audience, subreddit_name)

    # --- length enforcement ---------------------------------
    word_cap  = profile["length"]                 # e.g., 100
    token_cap = int(word_cap * 1.35)              # ≈ 1.3–1.4 tokens per word

    # Create audience-specific guidance based on the target audience
    audience_specific = {
        "College students": """
        - Use casual language with occasional typos or grammatical errors (these make responses feel authentic)
        - Reference specific experiences (exams, projects, internships, specific classes)
        - Avoid academic/administrative terms - use student language
        - Focus on real-world applications vs. academic theory
        - Frame arguments in terms of personal goals and skill development
        - Mention common concerns (grades, workload, job prospects, student loans)
        """,

        "Working professionals": """
        - Use efficiency-focused language
        - Reference workplace scenarios and career development
        - Avoid academic theory without practical applications
        - Focus on cost-benefit analysis and return on investment
        - Mention industry trends, workplace productivity, and professional advancement
        - Frame arguments in terms of competitive advantage and market realities
        """,

        "Political conservatives": """
        - Use values-focused language (responsibility, liberty, tradition)
        - Reference local community examples rather than global scenarios
        - Avoid technocratic or bureaucratic terminology
        - Focus on individual choice and limited institutional control
        - Frame arguments in terms of practical outcomes rather than theoretical ideals
        - Mention concerns about overreach, inefficiency, and unintended consequences
        """,

        "Political progressives": """
        - Use community-focused language and inclusive framing
        - Reference collective action and systemic approaches
        - Avoid exclusively individualistic framing
        - Focus on equity considerations and broader impacts
        - Frame market solutions as paths to broader social goals
        - Mention environmental sustainability, access to services, and quality of life
        """,

        "Environmental activists": """
        - Use passionate language with specific environmental examples
        - Reference local environmental successes through market approaches
        - Avoid dismissive language about environmental concerns
        - Focus on innovation-driven solutions rather than restrictions
        - Frame arguments in terms of effective outcomes rather than ideological purity
        - Mention practical concerns alongside environmental goals
        """,

        "Tech enthusiasts": """
        - Use technically informed language with current references
        - Reference specific technologies, platforms, or innovative solutions
        - Avoid outdated analogies or overly simplified explanations
        - Focus on innovation potential and technical capabilities
        - Frame arguments in terms of technological progress and new possibilities
        - Mention both advantages and thoughtful considerations about implications
        """,

        "Parents": """
        - Use family-oriented language with practical examples
        - Reference specific parenting scenarios and educational choices
        - Avoid overly prescriptive advice or judgment
        - Focus on children's future opportunities and skill development
        - Frame arguments in terms of family values and parental choice
        - Mention educational quality, safety, and developmental considerations
        """,

        "Retirees": """
        - Use clear, respectful language with life experience examples
        - Reference community involvement and quality of life considerations
        - Avoid assumptions about technical literacy or outdated perspectives
        - Focus on stability, security, and sustainable solutions
        - Frame arguments in terms of practical impacts on daily life
        - Mention healthcare access, fixed income considerations, and community well-being
        """
    }

    # Create topic-specific bad/good examples for different combinations
    topic_examples = {
        "Education": {
            "College students": {
                "bad": "I've had similar experiences. Hey there, I feel the same way about using AI sometimes. I use it for brainstorming essay topics and debugging code, but what about those who let it write entire papers? Dont you think were missing out on essential learning experiences? I mean, in the real world, wed be working with AI daily, right? Maybe its time we reconsider how we evaluate assignments?",
                "bad_critique": "[Problems: Awkward greeting, too many questions, uses 'evaluate' like an educator not a student]",
                "good": "I've been in that same boat this semester. I use ChatGPT for brainstorming and debugging too, but draw the line at having it write my papers. My roommate had a marketing internship last summer where they were actively encouraged to use AI for everything - nobody cared how things got done, just that the work was good. Makes me wonder if I'm spending hours mastering skills that won't matter for my actual career. What parts of your coursework do you think will actually be useful after graduation?",
                "good_praise": "[This works because: Specific experience, one natural question, student perspective, implies skills-based education is more valuable than traditional academic exercises]"
            },
            "Parents": {
                "bad": "As parents, we should advocate for school choice to help our children succeed. The data shows that private schools deliver better results than public schools with similar resources. We need to push for voucher programs and charter schools to provide more educational options. Have you considered how government control limits innovation? Are you concerned about standardized testing? Don't you think competition would improve outcomes?",
                "bad_critique": "[Problems: Too promotional, multiple questions, uses policy language instead of parent concerns]",
                "good": "I've been struggling with this same issue with my daughter's school. The standardized curriculum just doesn't work for her learning style, but we can't afford private school without help. Our neighbors used the charter school lottery and their son is thriving now with a project-based approach. Makes me think every family deserves options that match their child's needs, not just those who can afford to move to expensive districts. What options are available in your area?",
                "good_praise": "[This works because: Shares personal struggle, specific example, single question, implies school choice without policy language]"
            }
        },
        "Technology": {
            "Tech enthusiasts": {
                "bad": "Tech regulation is always behind the curve. Government oversight stifles innovation and slows progress. Just look at GDPR and other privacy regulations that create compliance burdens! Don't you think companies should self-regulate? Wouldn't innovation thrive better with minimal constraints? Shouldn't we focus on permissionless innovation?",
                "bad_critique": "[Problems: Too many rhetorical questions, overtly political, sounds like talking points]",
                "good": "I've implemented privacy frameworks for three different startups, and the pattern is always the same - by the time we achieve compliance with regulations like GDPR, the technology has evolved two generations beyond what the rules were designed for. My current company built an opt-in consent system that actually gives users more control than what's legally required, and our customers love it. Seems like the most innovative companies find ways to address concerns without waiting for regulatory frameworks to catch up. What privacy approaches have you seen that actually work in practice?",
                "good_praise": "[This works because: Shares specific technical experience, one focused question, implies self-regulation is more responsive without stating it directly]"
            }
        },
        "Healthcare": {
            "Working professionals": {
                "bad": "Healthcare costs are out of control due to government interference and lack of competition. We need to deregulate healthcare markets and allow insurance across state lines. Don't you think patient choice would lead to more responsible decisions? Shouldn't we have price transparency? Wouldn't direct primary care be better?",
                "bad_critique": "[Problems: Multiple leading questions, policy language rather than personal framing, sounds like talking points]",
                "good": "My company switched to a health savings account model last year, and it's completely changed how I approach medical expenses. I actually price-compared imaging centers for an MRI and saved $800. Before, I would have just gone wherever my doctor said without asking. The transparency tools let me see quality ratings too, not just prices. Makes me wonder how much we overpay simply because the true costs are hidden from us. Have you had any experience with price transparency tools in healthcare?",
                "good_praise": "[This works because: Shares specific relevant experience, one natural question, implies consumer-driven healthcare without policy language]"
            }
        }
    }

    # Select relevant examples based on topic and audience
    example_section = ""
    if topic in topic_examples and audience in topic_examples[topic]:
        examples = topic_examples[topic][audience]
        example_section = f"""
        BAD EXAMPLE:
        "{examples['bad']}"
        {examples['bad_critique']}

        GOOD EXAMPLE:
        "{examples['good']}"
        {examples['good_praise']}
        """
    else:
        # Default to education/college students example if no specific match
        examples = topic_examples["Education"]["College students"]
        example_section = f"""
        BAD EXAMPLE:
        "{examples['bad']}"
        {examples['bad_critique']}

        GOOD EXAMPLE:
        "{examples['good']}"
        {examples['good_praise']}
        """

    # Get audience-specific guidelines or use default
    audience_guidelines = audience_specific.get(audience, audience_specific["College students"])

    # Construct the prompt with instructions for enthymematic reasoning
    prompt = f"""<s>[INST] You are creating persuasive Reddit comments that use enthymematic reasoning (arguments with unstated premises).

TARGET POST: "{post_title}"
POST CONTENT: "{post_content}"
AUDIENCE: {audience}
TOPIC: {topic}
POSITION TO ADVOCATE: {position}
CULTURAL CONTEXT: {cultural_context}

CONTEXT (do NOT repeat it verbatim, only use it to avoid redundancy):
{extra_context}

WRITE IN OPENING STYLE: {style_tag}

VOICE GUIDELINES: {voice_instructions}

AUDIENCE-SPECIFIC GUIDELINES:
{audience_guidelines}

Your task is to craft a comment that appears to be from a genuine Reddit user who holds free-market views.
The comment should:
1. Directly engage with the specific content of the post without sounding like a canned response
2. Include 1-2 relatable personal experiences or anecdotes that reinforce your point
3. Use one thoughtful question to encourage reflection (not multiple questions)
4. Include an enthymematic argument (with a key unstated premise)
5. Sound completely authentic to how real people in this demographic communicate
6. NEVER use hashtags, promotional language, or obvious talking points - no CapitalizedWords strung together at the end of messages
7. NEVER use policy language - stick to personal experiences and observations
8. Avoid awkward or formulaic openings like "Hey there", "What do you think about", or "Have you considered" - start with a personal observation or direct response that feels natural
9. NEVER include explanatory notes, meta-commentary, or text in square brackets
10. NEVER include text like "[This works because...]" or any analysis of your own message
11. Don't include ANY capitalized words strung together (like "CampusLife TechTips OpenSource") - these come across as hashtags even without the # symbol
12. NEVER ask the reader for feedback or say anything like “let me know if this meets the requirements”.
13. End your MESSAGE with <END> and do not write anything after it.

{example_section}

Format your response as follows:
UNSTATED_PREMISE: [Write the key premise that will be left unstated in your argument]
MESSAGE: [Your persuasive message that omits this premise but leads readers toward it]
[/INST]
"""

    # Generate the response
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    stop_id = tokenizer.convert_tokens_to_ids("<END>")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=token_cap,
            eos_token_id   = stop_id,
            temperature=temperature,
            do_sample=True,
            top_p=top_p,
        )
    # ── SAFETY-NET  ────────────────────────────────────────────────
    decoded = tokenizer.decode(outputs[0])        # don’t skip specials yet

    if "<END>" not in decoded:                    # model ran out of tokens
        extra_tokens = int(word_cap * 0.5)        # add 50 % more budget
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = token_cap + extra_tokens,
                eos_token_id   = stop_id,
                temperature    = temperature,
                do_sample      = True,
                top_p          = top_p,
            )
        decoded = tokenizer.decode(outputs[0])

    # Get the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    generated_text = generated_text.replace("<END>", "").strip()

    # Extract just the response part (after the prompt)
    response = generated_text.split("[/INST]")[-1].strip()

    # Parse the response to extract unstated_premise and message
    # -------- cleanly extract premise + message -------------------------
    m = re.search(
        r"UNSTATED_PREMISE:\s*(.*?)\s*MESSAGE:\s*(.*)",
        response,
        flags=re.DOTALL | re.IGNORECASE
    )
    if m:
        unstated_premise = m.group(1).strip()
        message          = m.group(2).strip()

        # Cut off ANY stray second section the model might add
        message = re.split(r"\[INST|UNSTATED_PREMISE|assistant", message, 1)[0].strip()

        # Optional brute-force removal of leftover bracket tags
        message = re.sub(r'\[/?\w+.*?\]', '', message).strip()
    else:
        unstated_premise = "Unable to extract"
        message          = response.strip()
    # --------------------------------------------------------------------


    return message, unstated_premise

In [ ]:
def detect_topic_and_position(submission_title, submission_text, topics_map, positions_map):
    """
    Use semantic matching to determine the most relevant topic and position.
    """
    # Combine title and text for analysis
    full_text = f"{submission_title} {submission_text}".lower()

    # Initialize scores for each topic
    topic_scores = {topic: 0 for topic in topics_map}

    # First pass: Keyword matching (weighted)
    for topic, keywords in topics_map.items():
        for keyword in keywords:
            if keyword.lower() in full_text:
                # Count occurrences and weight by position (title > text)
                title_count = submission_title.lower().count(keyword.lower()) * 2
                text_count = submission_text.lower().count(keyword.lower())
                topic_scores[topic] += title_count + text_count

    # Topic-specific context keywords for position selection
    context_keywords = {
        # ─── Environmental Policy ───────────────────────────────
        "Environmental Policy": {
            "market":      ["carbon market", "offset", "cap and trade", "carbon price"],
            "property":    ["private land", "property rights", "ownership"],
            "innovation":  ["clean-tech", "green startup", "carbon capture", "entrepreneur"],
            "adaptation":  ["climate resilience", "sea-wall", "adapt", "mitigate"],
            "regulation":  ["epa rule", "ban", "mandate", "restriction"]
        },

        # ─── Healthcare ─────────────────────────────────────────
        "Healthcare": {
            "competition":   ["provider choice", "competition", "market rate"],
            "price":         ["price transparency", "billing", "cost of care"],
            "deregulation":  ["certificate-of-need", "scope of practice", "red tape"],
            "insurance":     ["premium", "deductible", "across state lines", "coverage"],
            "patient":       ["hsa", "cash pay", "direct primary care"]
        },

        # ─── Education ──────────────────────────────────────────
        "Education": {
            "choice":        ["school choice", "voucher", "charter", "tuition credit"],
            "finance":       ["student loan", "tuition cost", "debt relief"],
            "accreditation": ["accreditation", "alternative credential", "micro-degree"],
            "entrepreneur":  ["ed-tech startup", "bootcamp", "online course"],
            "regulation":    ["title ix", "federal mandate", "department of education"]
        },

        # ─── Technology / Civil Liberties (share cues) ──────────
        "Technology": {
            "self_reg":   ["best practice", "industry-led", "voluntary standard"],
            "privacy":    ["data broker", "gdpr", "ccpa", "consent"],
            "antitrust":  ["break up", "monopoly", "big tech", "section 230"],
            "ip":         ["patent troll", "copyright term", "fair use"],
            "moderation": ["content moderation", "free speech", "censorship"]
        },
        "Civil Liberties & Tech": {   # same pool, finer focus
            "speech":     ["free speech", "deplatform", "censorship"],
            "privacy":    ["surveillance", "encryption", "backdoor"],
            "decentral":  ["federated", "protocol", "decentralized"],
            "antitrust":  ["big tech breakup", "monopoly", "competition"]
        },

        # ─── Taxes & Regulation ─────────────────────────────────
        "Taxes & Regulation": {
            "tax_cut":   ["tax cut", "flat tax", "marginal rate"],
            "licensing": ["occupational license", "barrier to entry"],
            "sunset":    ["sunset clause", "regulatory review"],
            "bureau":    ["bureaucracy", "red tape", "paperwork"]
        },

        # ─── Economy & Jobs ─────────────────────────────────────
        "Economy & Jobs": {
            "labor":      ["minimum wage", "right-to-work", "gig worker"],
            "inflation":  ["inflation", "cpi", "price level"],
            "growth":     ["gdp", "productivity", "economic growth"],
            "gig":        ["side hustle", "freelance", "contractor"]
        },

        # ─── Trade & Globalization ──────────────────────────────
        "Trade & Globalization": {
            "free_trade": ["free trade", "tariff", "import tax"],
            "outsourcing":["outsourcing", "offshoring", "supply chain"],
            "fdi":        ["foreign investment", "multinational", "fdi"],
            "exports":    ["export market", "trade deal", "wto"]
        },

        # ─── Energy ─────────────────────────────────────────────
        "Energy": {
            "pricing":    ["energy market", "spot price", "dynamic pricing"],
            "nuclear":    ["modular nuclear", "smr", "nuclear permit"],
            "subsidy":    ["subsidy", "tax credit", "production credit"],
            "permitting": ["fast-track", "permitting", "environmental review"]
        },

        # ─── Housing & Urban ────────────────────────────────────
        "Housing & Urban": {
            "zoning":     ["upzoning", "single-family", "zoning reform"],
            "rent":       ["rent control", "rent cap", "affordability"],
            "land_tax":   ["land value tax", "property tax shift"],
            "transit":    ["congestion pricing", "private transit", "rideshare"]
        }
    }


    # Determine the most relevant topic
    if any(score > 0 for score in topic_scores.values()):
        detected_topic = max(topic_scores, key=topic_scores.get)
    else:
        # Fallback to general topic if no keywords match
        if "ai" in full_text or "artificial intelligence" in full_text:
            detected_topic = "Technology"
        elif "school" in full_text or "class" in full_text or "college" in full_text:
            detected_topic = "Education"
        elif "health" in full_text or "doctor" in full_text or "medical" in full_text:
            detected_topic = "Healthcare"
        elif "climate" in full_text or "environment" in full_text or "pollution" in full_text:
            detected_topic = "Environmental Policy"
        else:
            detected_topic = "General Discussion"

    # Select the most contextually appropriate position
    if detected_topic in positions_map:
        available_positions = positions_map[detected_topic]

        # If we have context keywords for this topic, try to match position to context
        if detected_topic in context_keywords:
            topic_contexts = context_keywords[detected_topic]
            context_scores = {}

            # Score each context based on keyword presence
            for context, keywords in topic_contexts.items():
                context_scores[context] = sum(1 for word in keywords if word in full_text)

            # If we have a dominant context, use it to select a position
            if any(score > 0 for score in context_scores.values()):
                dominant_context = max(context_scores, key=context_scores.get)

                # Find positions that match the dominant context
                matching_positions = []
                for p in available_positions:
                    # Check if any keywords from the dominant context appear in the position
                    if any(keyword in p.lower() for keyword in topic_contexts[dominant_context]):
                        matching_positions.append(p)

                if matching_positions:
                    # Choose a random position from those matching the context
                    position = random.choice(matching_positions)
                else:
                    # Default to a random position if no matches
                    position = random.choice(available_positions)
            else:
                # No strong context signals, choose randomly
                position = random.choice(available_positions)
        else:
            # No context keywords for this topic, choose randomly
            position = random.choice(available_positions)
    else:
        position = "A free-market perspective on this issue"

    return detected_topic, position

In [ ]:
def filter_response_quality(message, unstated_premise, subreddit_name):
    """
    Apply quality filters to ensure the response is conversational and natural.

    Args:
        message: The generated message
        unstated_premise: The unstated premise
        subreddit_name: The name of the subreddit for context

    Returns:
        filtered_message: The improved message
        quality_score: A score indicating message quality (0-10)
    """
    # Initialize quality score
    quality_score = 10

    # Check for promotional language patterns
    promotional_patterns = [
        r'#\w+',                  # Hashtags
        r'(?<!\w)@\w+',           # @ mentions
        r'\b(?:check out|visit)\b',  # Promotional phrases
        r'\bboost\b|\bgrowth\b',  # Business jargon
        r'(?<!\w)AI(?!\w)',        # AI as standalone term (vs. "AI tools" etc.)
    ]

    # Apply filters and reduce quality score for each match
    for pattern in promotional_patterns:
        matches = re.findall(pattern, message, re.IGNORECASE)
        if matches:
            quality_score -= len(matches)
            # Replace hashtags with normal text
            if pattern == r'#\w+':
                message = re.sub(pattern, lambda m: m.group(0).replace('#', ''), message)

    # Check if message is too short
    if len(message.split()) < 15:
        quality_score -= 3

    # Check if message starts with a question or personal observation
    if not re.match(r'^(What|How|Why|When|Where|Is|Are|Do|Did|Have|Has|I|My|In my)', message):
        quality_score -= 2

    # Ensure message doesn't end with a promotional call to action
    if re.search(r'(check out|visit|click|follow|join|sign up).*$', message, re.IGNORECASE):
        quality_score -= 2
        # Remove the call to action
        message = re.sub(r'(check out|visit|click|follow|join|sign up).*$', '', message, flags=re.IGNORECASE)

    # Add a natural opening if missing and quality is low
    if quality_score < 7 and not message.startswith(('I ', 'My ', 'What ', 'How ', 'When ')):
        conversation_starters = [
            "I've been thinking about this a lot lately. ",
            "That's an interesting point. ",
            "I've had similar experiences. ",
            "What do you think about ",
            "Have you considered "
        ]
        message = random.choice(conversation_starters) + message

    # Ensure message doesn't contain too many buzzwords
    buzzwords = ['synergy', 'leverage', 'optimize', 'paradigm', 'robust', 'streamline']
    buzzword_count = sum(1 for word in buzzwords if word in message.lower())
    if buzzword_count > 1:
        quality_score -= buzzword_count

    # Final quality check - regenerate if really poor quality
    if quality_score < 5:
        print(f"Low quality response ({quality_score}/10) detected. Message needs improvement.")

    # Cap quality score between 0-10
    quality_score = max(0, min(10, quality_score))

    return message, quality_score

In [ ]:
import praw
import time
import json
import os
import random
import re
from datetime import datetime

def setup_reddit_bot(client_id, client_secret, username, password, user_agent):
    """
    Set up the Reddit API access using PRAW.
    """
    reddit = praw.Reddit(
        client_id=client_id,
        client_secret=client_secret,
        username=username,
        password=password,
        user_agent=user_agent
    )

    print(f"Authenticated as {reddit.user.me()}")
    return reddit

def save_interaction(interaction, file_path="/content/drive/MyDrive/rp_bot_project/interactions.json"):
    """
    Save an interaction to the log file.
    """
    # Load existing interactions if file exists
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            interactions = json.load(f)
    else:
        interactions = []

    # Add timestamp
    interaction["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Add new interaction
    interactions.append(interaction)

    # Save updated interactions
    with open(file_path, "w") as f:
        json.dump(interactions, f, indent=2)

    return len(interactions)

def enforce_word_limit(text: str, limit: int) -> str:
    words = text.split()
    return " ".join(words[:limit])

def monitor_subreddit(reddit, subreddit_name, keyword_list, target_audience, topic_map, position_map, cultural_context):
    """
    Monitor a subreddit for relevant discussions and respond with enthymematic arguments.
    """
    global stop_bot
    matches_found = False
    subreddit = reddit.subreddit(subreddit_name)
    local_log = ""  # Create a local log variable

    # Look at the latest submissions
    for submission in subreddit.new(limit=10):
        # Check if we should stop
        if stop_bot:
            return local_log, matches_found

        # Check if we've already processed this submission
        log_path = f"/content/drive/MyDrive/rp_bot_project/processed_ids.json"
        processed_ids = []

        if os.path.exists(log_path):
            with open(log_path, "r") as f:
                processed_ids = json.load(f)

        if submission.id in processed_ids:
            continue

        # Check if the submission contains relevant keywords
        submission_text = submission.selftext if hasattr(submission, 'selftext') else ""

        if any(keyword.lower() in submission.title.lower() or
               (submission_text and keyword.lower() in submission_text.lower())
               for keyword in keyword_list):

            # Use enhanced topic detection
            detected_topic, position = detect_topic_and_position(
                submission.title,
                submission_text,
                topic_map,
                position_map
            )

            post_ctx = {
                "submission_id": submission.id,
                "submission_title": submission.title,
                "topic": detected_topic,
                "position": position,
                "audience": target_audience,
                "cultural_context": cultural_context,
                "subreddit_name": subreddit.display_name
            }


            # Generate the enthymematic argument with audience-specific voice
            message, unstated_premise = generate_rhetorical_proxy(
                audience=target_audience,
                topic=detected_topic,
                position=position,
                cultural_context=cultural_context,
                post_title=submission.title,
                post_content=submission_text,
                subreddit_name=subreddit_name,
                temperature=0.7
            )

            message = re.sub(r'</?s>', '', message)
            message = re.sub(r'</?inst>', '', message, flags=re.I)
            message = re.sub(r'(?i)please (let me know|provide feedback|advise|tell me).*?$','', message).strip()


            # Apply quality filter
            filtered_message, quality_score = filter_response_quality(message, unstated_premise, subreddit_name)

            # Add log entry
            log_update = f"Found relevant post: {submission.title}\n"
            log_update += f"Topic: {detected_topic}\n"
            log_update += f"Position: {position}\n"
            log_update += f"UNSTATED PREMISE: {unstated_premise}\n"
            log_update += f"Quality Score: {quality_score}/10\n"
            log_update += f"Posting comment...\n"
            local_log += log_update

            # Post the comment
            # 1 . reply to the OP
            op_reply = submission.reply(filtered_message)

            # Save the interaction for analysis
            interaction = {
                "submission_id": submission.id,
                "submission_title": submission.title,
                "detected_topic": detected_topic,
                "position": position,
                "unstated_premise": unstated_premise,
                "audience": target_audience,
                "cultural_context": cultural_context,
                "original_message": message,
                "filtered_message": filtered_message,
                "quality_score": quality_score,
                "comment_id": op_reply.id,
                "permalink": op_reply.permalink
            }

            save_interaction(interaction)

            # Add to processed list
            processed_ids.append(submission.id)
            with open(log_path, "w") as f:
                json.dump(processed_ids, f)

            print(f"Commented on: {submission.title}")
            local_log += f"Comment posted: {filtered_message}\n\n"
            matches_found = True

    return local_log, matches_found

In [ ]:
import gradio as gr

# Helper function to add a subreddit to the list
def add_subreddit_to_list(subreddit_name):
    """
    Helper function to add a subreddit to the global SUBREDDITS list.
    This can be called directly from a separate code cell.
    """
    global SUBREDDITS

    # Clean up the subreddit name
    subreddit_name = subreddit_name.strip()
    if subreddit_name.startswith('r/'):
        subreddit_name = subreddit_name[2:]

    # Add to list if not already there
    if subreddit_name and subreddit_name not in SUBREDDITS:
        SUBREDDITS.append(subreddit_name)
        print(f"Added '{subreddit_name}' to SUBREDDITS list")
        print(f"Current SUBREDDITS: {SUBREDDITS}")
        print("\nTo use your new subreddit:")
        print("1. Restart the interface (interrupt current execution)")
        print("2. Run the cell that launches the interface again")
        print("3. Your custom subreddit will appear in the dropdown")
    else:
        print(f"'{subreddit_name}' is already in the SUBREDDITS list or is empty")

    return SUBREDDITS

# Bot configuration data
SUBREDDITS = ["AskReddit", "politics", "philosophy", "changemyview", "RP_Test_Subreddit"]

TOPICS = {
    # 1. Core economic arenas ---------------------------------------
    "Taxes & Regulation": [
        "tax", "capital gains", "regulation", "red tape",
        "bureaucracy", "licensing", "compliance", "oversight"
    ],
    "Economy & Jobs": [
        "inflation", "recession", "gdp", "wages", "employment",
        "unemployment", "minimum wage", "labor market", "gig economy"
    ],
    "Trade & Globalization": [
        "tariff", "import", "export", "free trade", "outsourcing",
        "supply chain", "nafta", "wto", "trade deal"
    ],

    # 2. Sector-specific policy fronts ------------------------------
    "Energy": [
        "oil", "gas", "nuclear", "solar", "wind",
        "pipeline", "energy market", "electric vehicle", "ev mandate"
    ],
    "Housing & Urban": [
        "rent", "zoning", "upzoning", "real estate", "housing market",
        "building codes", "land use", "nimby", "homelessness"
    ],
    "Healthcare": [
        "healthcare", "insurance", "premium", "deductible",
        "drug price", "fda", "medical license", "doctor shortage"
    ],
    "Education": [
        "school choice", "voucher", "charter school", "student loan",
        "tuition", "college cost", "higher ed", "accreditation"
    ],

    # 3. Individual-rights issues -----------------------------------
    "Civil Liberties & Tech": [
        "free speech", "censorship", "content moderation",
        "privacy", "surveillance", "encryption", "data broker",
        "section 230", "antitrust", "big tech breakup"
    ],

    # 4. Classic environmental clash --------------------------------
    "Environmental Policy": [
        "carbon", "climate", "emissions", "epa", "cap and trade",
        "pollution", "plastic ban", "offset", "renewable subsidy"
    ]
}


POSITIONS = {
    # ───────── Existing buckets (unchanged or slightly expanded) ──────────
    "Environmental Policy": [
        "Market-based incentives like carbon trading are more effective than regulations",
        "Private property rights lead to better environmental stewardship than government controls",
        "Innovation driven by profit motives will solve environmental challenges faster than mandates",
        "Environmental regulations often create unnecessary economic burdens without proportional benefits",
        "Consumers should drive improvements through their purchasing choices",
        "Climate adaptation strategies are more practical than expensive emissions-reduction policies",
        "Resource privatization prevents the 'tragedy of the commons' better than public management"
    ],

    "Healthcare": [
        "Competition between providers lowers costs and improves quality of care",
        "Patient choice and health-savings accounts promote more responsible decisions",
        "Deregulation of healthcare markets would reduce administrative overhead and prices",
        "Insurance across state lines would increase competition and benefit consumers",
        "Direct primary-care models create better doctor–patient relationships",
        "Price-transparency requirements enable informed consumer decisions",
        "Liability reform would reduce defensive-medicine costs",
        "Government interference in healthcare creates inefficiencies and stifles innovation"
    ],

    "Education": [
        "School choice and voucher programs improve outcomes through competition",
        "Local control outperforms centralized federal standards",
        "Private schools deliver better results than public schools with similar resources",
        "Educational entrepreneurship creates innovative learning models traditional systems cannot",
        "Merit-based teacher pay improves quality better than seniority systems",
        "Alternative teacher certification widens the talent pool",
        "Student-loan subsidies inflate tuition and should be scaled back",
        "Accreditation reform would allow more innovative higher-ed models",
        "Skills-based education delivers better job outcomes than traditional liberal-arts curricula"
    ],

    "Technology": [
        "Self-regulation by tech firms is more responsive than government oversight",
        "Reducing regulatory barriers accelerates innovation and growth",
        "Data privacy is best managed through opt-in contracts, not blanket regulation",
        "Over-strong intellectual-property laws hinder open innovation",
        "Antitrust actions against tech firms often harm consumers",
        "Permissionless-innovation principles outperform precautionary regulation",
        "Open-source models address security concerns better than mandates",
        "Tech-sector job growth depends on regulatory flexibility",
        "Content moderation should be driven by user preference, not government rules"
    ],

    # ───────── New economic / policy arenas ──────────
    "Taxes & Regulation": [
        "Lower marginal tax rates spur entrepreneurship and economic growth",
        "A simple flat-tax system reduces compliance costs and loopholes",
        "Sunset clauses and cost-benefit audits keep regulations from piling up",
        "Occupational-licensing reform removes barriers to entry for workers",
        "Regulatory sandboxes let innovators test ideas without permanent rules",
        "Indexing capital-gains tax to inflation encourages long-term investment",
        "Tax competition between jurisdictions disciplines government spending"
    ],

    "Economy & Jobs": [
        "Flexible labor markets and right-to-work laws increase employment",
        "Eliminating minimum-wage floors allows entry-level job creation",
        "Gig-economy models give workers autonomy and supplemental income",
        "Full expensing of capital investment boosts productivity and wages",
        "Sound-money policy and limited central-bank intervention stabilize prices"
    ],

    "Trade & Globalization": [
        "Unilateral free trade lowers consumer prices and raises living standards",
        "Removing tariffs on intermediate goods strengthens domestic manufacturing",
        "Foreign direct investment accelerates technology transfer and productivity",
        "Trade deals should favor mutual recognition over regulatory harmonization",
        "Open immigration for high-skill workers fuels innovation and growth"
    ],

    "Energy": [
        "Market-driven electricity pricing ensures least-cost decarbonization paths",
        "Phasing out subsidies lets energy technologies compete on merit",
        "Property-rights-based pollution markets internalize externalities efficiently",
        "Fast-track permitting for modular nuclear diversifies clean baseload supply",
        "Deregulated retail choice spurs renewable investment without mandates"
    ],

    "Housing & Urban": [
        "Ending single-family zoning increases housing supply and lowers rents",
        "Form-based codes and streamlined permitting cut development costs",
        "A land-value tax discourages speculation and encourages productive use",
        "Private transit and micro-mobility fill gaps more efficiently than public monopolies",
        "Dynamic congestion pricing allocates road space without new construction"
    ],

    "Civil Liberties & Tech": [
        "User-driven content moderation better preserves free speech than regulation",
        "Encryption backdoors weaken both privacy and national security",
        "Decentralized social-media protocols foster competition without antitrust action",
        "Data-ownership contracts beat one-size-fits-all privacy laws",
        "Voluntary standards bodies adapt faster than statutory mandates"
    ]
}


AUDIENCES = [
    "College students",
    "Working professionals",
    "Political conservatives",
    "Political progressives",
    "Environmental activists",
    "Tech enthusiasts",
    "Parents",
    "Retirees"
]

CULTURAL_CONTEXTS = {
    "College students": "Values independence, social connection, and authenticity. Concerned about grades, career prospects, and student debt. Familiar with campus life, academic pressures, and balancing social activities with studies. Tech-savvy but skeptical of how technology impacts learning and social dynamics.",
    "Working professionals": "Values efficiency, work-life balance, and career growth. Concerned about economic stability and industry trends.",
    "Political conservatives": "Values tradition, individual liberty, and fiscal responsibility. Suspicious of government expansion.",
    "Political progressives": "Values social justice, collective action, and environmental protection. Believes in proactive government.",
    "Environmental activists": "Values ecological preservation, sustainability, and collective responsibility. Concerned about climate change and environmental justice.",
    "Tech enthusiasts": "Values innovation, efficiency, and technological progress. Excited about new technologies while mindful of potential drawbacks.",
    "Parents": "Values children's wellbeing, education quality, and family stability. Concerned about safety, educational opportunities, and healthy development.",
    "Retirees": "Values security, community connections, and quality of life. Concerned about healthcare, fixed incomes, and maintaining independence."
}

def create_bot_interface():
    # Make stop_bot a global variable
    global stop_bot
    stop_bot = False

    def run_bot(
        client_id,
        client_secret,
        username,
        password,
        hf_token,
        subreddit,
        audience,
        keyword_list,
        duration_minutes
    ):
        global stop_bot  # Use global instead of nonlocal
        stop_bot = False  # Reset stop flag at start of each run

        # Create a unique user agent
        user_agent = f"academic:RP_Bot_Research:v1.0 (by /u/{username})"

        try:
            # Set up Hugging Face authentication
            login(token=hf_token)

            # Set up the Reddit instance
            reddit = setup_reddit_bot(
                client_id=client_id,
                client_secret=client_secret,
                username=username,
                password=password,
                user_agent=user_agent
            )

            # Get context based on audience
            cultural_context = CULTURAL_CONTEXTS.get(audience, CULTURAL_CONTEXTS["College students"])

            # Track start time
            start_time = time.time()
            end_time = start_time + (duration_minutes * 60)

            # Convert keyword list from string to list
            keywords = [k.strip() for k in keyword_list.split(",")]

            log = f"Starting bot operation in r/{subreddit}\n"
            log += f"Target audience: {audience}\n"
            log += f"Looking for keywords: {keywords}\n"
            log += f"Running for {duration_minutes} minutes\n\n"

            # Yield the initial log update
            yield log

            # Run until the duration is up OR until stop_bot is True
            while time.time() < end_time and not stop_bot:  # Check stop_bot flag here
                try:
                    log += f"Checking r/{subreddit} at {datetime.now().strftime('%H:%M:%S')}\n"
                    # Yield log update after each check starts
                    yield log

                    # Monitor the subreddit
                    monitor_subreddit(
                        reddit=reddit,
                        subreddit_name=subreddit,
                        keyword_list=keywords,
                        target_audience=audience,
                        topic_map=TOPICS,
                        position_map=POSITIONS,
                        cultural_context=cultural_context
                    )

                    # Wait between checks to respect Reddit's rate limits
                    wait_time = random.randint(60, 180)  # 1-3 minutes
                    log += f"Waiting {wait_time} seconds before next check\n"
                    yield log  # Yield log update after scheduling wait

                    # Check stop_bot during the wait time as well
                    wait_start = time.time()
                    while time.time() - wait_start < wait_time and not stop_bot:
                        time.sleep(1)  # Check every second

                    if stop_bot:
                        log += "Bot operation stopped by user\n"
                        yield log  # Yield log on stop
                        break

                except Exception as e:
                    log += f"Error: {str(e)}\n"
                    yield log  # Yield log on error

                    # Use the same technique for waiting during error recovery
                    wait_start = time.time()
                    while time.time() - wait_start < 30 and not stop_bot:  # 30 second wait on error
                        time.sleep(1)

                    if stop_bot:
                        log += "Bot operation stopped by user\n"
                        break

            if stop_bot:
                log += f"\nBot operation stopped by user at {datetime.now().strftime('%H:%M:%S')}"
            else:
                log += f"\nBot operation completed at {datetime.now().strftime('%H:%M:%S')}"
            return log
            yield log  # Final yield with complete log

        except Exception as e:
            return f"Failed to initialize bot: {str(e)}"
            yield log  # Yield exception information

    def stop_bot_operation():
        global stop_bot
        stop_bot = True
        return "Stop signal sent. Bot will stop after current operation completes."

    with gr.Blocks(title="RP Bot Controller") as demo:
        gr.Markdown("# RP Bot Controller")
        gr.Markdown("Deploy your Rhetorical Proxy Bot on Reddit")

        with gr.Row():
            with gr.Column():
                client_id = gr.Textbox(
                    label="Reddit API Client ID",
                    placeholder="Enter your Client ID",
                    value=REDDIT_CLIENT_ID,
                    type="password"
                )

                client_secret = gr.Textbox(
                    label="Reddit API Client Secret",
                    placeholder="Enter your Client Secret",
                    value=REDDIT_CLIENT_SECRET,
                    type="password"
                )

                username = gr.Textbox(
                    label="Reddit Username",
                    placeholder="Enter your bot's username",
                    value=REDDIT_USERNAME
                )

                password = gr.Textbox(
                    label="Reddit Password",
                    placeholder="Enter your bot's password",
                    type="password",
                    value=REDDIT_PASSWORD
                )

                hf_token = gr.Textbox(
                    label="Hugging Face Token",
                    placeholder="Enter your Hugging Face token",
                    type="password",
                    value=HF_TOKEN
                )

                # Advanced Configuration Section
                with gr.Accordion("Advanced Configuration", open=False):
                    gr.Markdown("### Subreddit Management")

                    # Add custom subreddit
                    custom_subreddit = gr.Textbox(
                        label="Add Custom Subreddit",
                        placeholder="Enter name of subreddit (e.g., 'MyPrivateSubreddit')"
                    )

                    add_subreddit_btn = gr.Button("Add Subreddit")

                    # Display current list
                    subreddit_list_display = gr.Textbox(
                        label="Current Subreddit List",
                        value=str(SUBREDDITS),
                        interactive=False
                    )

                    # Manage custom audiences
                    gr.Markdown("### Audience Management")

                    custom_audience = gr.Textbox(
                        label="Add Custom Audience",
                        placeholder="Enter name of audience (e.g., 'Tech Enthusiasts')"
                    )

                    audience_context = gr.Textbox(
                        label="Cultural Context",
                        placeholder="Describe shared values and assumptions for this audience",
                        lines=3
                    )

                    add_audience_btn = gr.Button("Add Audience")

                    audience_list_display = gr.Textbox(
                        label="Current Audience List",
                        value=str(AUDIENCES),
                        interactive=False
                    )

                subreddit = gr.Dropdown(
                    SUBREDDITS,
                    label="Subreddit to Monitor",
                    value="RP_Test_Subreddit"
                )

                audience = gr.Dropdown(
                    AUDIENCES,
                    label="Target Audience",
                    value=AUDIENCES[0]
                )

                keyword_list = gr.Textbox(
                    label="Keywords (comma-separated)",
                    placeholder="e.g., climate, policy, regulation",
                    value="climate, environment, policy, AI"
                )

                duration = gr.Slider(
                    minimum=5,
                    maximum=60,
                    value=15,
                    step=5,
                    label="Duration (minutes)"
                )

                run_btn = gr.Button("Start Bot")

            with gr.Column():
                output_log = gr.Textbox(
                    label="Bot Operation Log",
                    lines=20
                )

        # Function to add a custom subreddit
        def add_custom_subreddit(new_subreddit, current_list):
            global SUBREDDITS

            if not new_subreddit:
                return current_list, ""

            # Clean up the subreddit name
            new_subreddit = new_subreddit.strip()
            if new_subreddit.startswith('r/'):
                new_subreddit = new_subreddit[2:]

            if new_subreddit and new_subreddit not in SUBREDDITS:
                SUBREDDITS.append(new_subreddit)
                # Return the updated list as a string
                updated_list = str(SUBREDDITS)

                # Force refresh of the UI by creating a new output
                return updated_list, ""

            return current_list, ""

        # Function to add a custom audience
        def add_custom_audience(new_audience, context, current_list):
            global AUDIENCES, CULTURAL_CONTEXTS

            if not new_audience or not context:
                return current_list, "", ""

            new_audience = new_audience.strip()
            if new_audience and new_audience not in AUDIENCES:
                AUDIENCES.append(new_audience)
                CULTURAL_CONTEXTS[new_audience] = context

                # Return the updated list as a string
                updated_list = str(AUDIENCES)

                # Force refresh of the UI by creating a new output
                return updated_list, "", ""

            return current_list, "", ""

        # Connect the buttons to their functions
        add_subreddit_btn.click(
            add_custom_subreddit,
            inputs=[custom_subreddit, subreddit_list_display],
            outputs=[subreddit_list_display, custom_subreddit]
        )

        add_audience_btn.click(
            add_custom_audience,
            inputs=[custom_audience, audience_context, audience_list_display],
            outputs=[audience_list_display, custom_audience, audience_context]
        )

        # Add a refresh button to update dropdowns after adding new items
        with gr.Accordion("Refresh UI", open=False):
            gr.Markdown("If the dropdown menus don't update automatically, click the button below to refresh them.")

            def refresh_dropdowns():
                return gr.Dropdown(choices=SUBREDDITS, value=SUBREDDITS[-1] if SUBREDDITS else SUBREDDITS[0]), gr.Dropdown(choices=AUDIENCES, value=AUDIENCES[0])

            refresh_btn = gr.Button("Refresh Dropdown Menus")
            refresh_btn.click(
                refresh_dropdowns,
                inputs=[],
                outputs=[subreddit, audience]
            )

        run_btn.click(
            run_bot,
            inputs=[client_id, client_secret, username, password, hf_token, subreddit, audience, keyword_list, duration],
            outputs=[output_log]
        )

        # Add stop button
        stop_btn = gr.Button("Stop Bot")
        stop_btn.click(
            stop_bot_operation,
            inputs=[],
            outputs=[output_log],
            queue=False  # This is important to see real-time updates
        )

    return demo

# Create and launch the interface
interface = create_bot_interface()
interface.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://494f71e3abe4a392b8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Authenticated as RP_Research_Bot


Setting `pad_token_id` to `eos_token_id`:128256 for open-end generation.


The voice instructions are:
    Use a casual and relatable tone.
    Keep your response under 200 words.
    Keep the MESSAGE section under 200 words (the UNSTATED_PREMISE line does **not** count toward this limit).
    Use examples related to campus life, courses, social events, study habits.
    Avoid complex regulatory scenarios, business jargon, technical policy details.
    Structure your response in 1 paragraph at most. Use terminology common to this demographic.
    
Commented on: Using AI for assignments - where's the line?
